In [2]:
!pip install -q gradio pandas scikit-learn plotly

In [3]:
import sqlite3
import pandas as pd
from datetime import datetime

# Connect to database
conn = sqlite3.connect("smartfix.db", check_same_thread=False)
cursor = conn.cursor()

# Create Users Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    user_id TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT NOT NULL
)
""")

# Create Complaints Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS complaints (
    complaint_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    category TEXT,
    description TEXT NOT NULL,
    location TEXT NOT NULL,
    priority TEXT,
    status TEXT DEFAULT 'Pending',
    image TEXT,
    date TEXT
)
""")

# Create default authority account
cursor.execute("""
INSERT OR IGNORE INTO users (name, user_id, password, role)
VALUES ('Admin', 'admin', 'admin123', 'authority')
""")

conn.commit()

print("SmartFix database created successfully!")
print("User table created.")
print("Complaints table created.")
print("Default authority account created.")

SmartFix database created successfully!
User table created.
Complaints table created.
Default authority account created.


In [4]:
import gradio as gr
import hashlib

# Password security
def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


# Citizen Registration
def register(name, user_id, password):
    if not name or not user_id or not password:
        return "Please fill in all fields."

    if len(password) < 4:
        return "Password must contain at least 4 characters."

    try:
        cursor.execute("""
        INSERT INTO users (name, user_id, password, role)
        VALUES (?, ?, ?, 'citizen')
        """, (name, user_id, hash_password(password)))

        conn.commit()
        return "Registration successful! You can now login."

    except sqlite3.IntegrityError:
        return "This User ID is already registered."


# User Login
def login(user_id, password):
    if not user_id or not password:
        return "Please enter your User ID and password."

    cursor.execute(
        "SELECT name, user_id, password, role FROM users WHERE user_id=?",
        (user_id,)
    )

    user = cursor.fetchone()

    if not user:
        return "Invalid User ID or password."

    name, uid, stored_password, role = user

    # Support the default admin account created earlier
    valid = (
        stored_password == hash_password(password)
        or (uid == "admin" and stored_password == password)
    )

    if not valid:
        return "Invalid User ID or password."

    if role == "authority":
        return f"Welcome {name}! Authority login successful."

    return f"Welcome {name}! Citizen login successful."


# SmartFix Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏙️ SmartFix
    ### AI-Powered Public Issue Reporting System
    Report public problems. Help create cleaner and safer communities.
    """)

    with gr.Tab("Citizen Registration"):

        gr.Markdown("### Create Your Account")

        reg_name = gr.Textbox(label="Full Name")
        reg_id = gr.Textbox(label="Create User ID")
        reg_password = gr.Textbox(
            label="Create Password",
            type="password"
        )

        reg_button = gr.Button("Register", variant="primary")
        reg_output = gr.Textbox(label="Registration Status")

        reg_button.click(
            register,
            inputs=[reg_name, reg_id, reg_password],
            outputs=reg_output
        )

    with gr.Tab("Login"):

        gr.Markdown("### Login to SmartFix")

        login_id = gr.Textbox(label="User ID")
        login_password = gr.Textbox(
            label="Password",
            type="password"
        )

        login_button = gr.Button("Login", variant="primary")
        login_output = gr.Textbox(label="Login Status")

        login_button.click(
            login,
            inputs=[login_id, login_password],
            outputs=login_output
        )

    gr.Markdown("""
    ---
    **For Authorities**

    Demo credentials:
    - User ID: admin
    - Password: admin123
    """)


demo.launch(share=True)

/tmp/ipykernel_1130/3486655114.py:63: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a720fddf19a5eb7dd5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
import gradio as gr
import sqlite3
import hashlib
import pandas as pd
import os
import shutil
from datetime import datetime

# Connect to existing database
conn = sqlite3.connect("smartfix.db", check_same_thread=False)
cursor = conn.cursor()

os.makedirs("uploads", exist_ok=True)


# Password hashing
def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()


# Login function
def user_login(user_id, password):

    if not user_id or not password:
        return None, "Please enter all details."

    cursor.execute(
        "SELECT name, user_id, password, role FROM users WHERE user_id=?",
        (user_id,)
    )

    user = cursor.fetchone()

    if not user:
        return None, "Invalid login details."

    name, uid, stored_password, role = user

    valid = (
        stored_password == hash_password(password)
        or (uid == "admin" and stored_password == password)
    )

    if not valid:
        return None, "Incorrect password."

    return {
        "name": name,
        "user_id": uid,
        "role": role
    }, f"Welcome {name}! Login successful."


# Registration
def user_register(name, user_id, password):

    if not name or not user_id or not password:
        return "Please fill all fields."

    if len(password) < 4:
        return "Password must contain at least 4 characters."

    try:
        cursor.execute("""
        INSERT INTO users (name, user_id, password, role)
        VALUES (?, ?, ?, 'citizen')
        """, (name, user_id, hash_password(password)))

        conn.commit()

        return "Registration successful! Please login."

    except sqlite3.IntegrityError:
        return "User ID already exists."


# Submit complaint
def submit_complaint(user, category, description, location, image):

    if not user or user["role"] != "citizen":
        return "Please login as a citizen first."

    if not category or not description or not location:
        return "Please complete all complaint details."

    image_path = None

    if image:
        extension = os.path.splitext(image)[1]
        filename = f"{user['user_id']}_{datetime.now().strftime('%Y%m%d%H%M%S%f')}{extension}"
        image_path = os.path.join("uploads", filename)
        shutil.copy(image, image_path)

    cursor.execute("""
    INSERT INTO complaints
    (user_id, category, description, location, priority, status, image, date)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        user["user_id"],
        category,
        description,
        location,
        "Pending AI Analysis",
        "Pending",
        image_path,
        datetime.now().strftime("%Y-%m-%d %H:%M")
    ))

    conn.commit()

    complaint_id = cursor.lastrowid

    return f"""
    Complaint submitted successfully!

    Your Complaint ID: SF-{complaint_id:04d}

    Category: {category}
    Status: Pending

    Save your Complaint ID to track your issue.
    """


# View citizen complaints
def view_my_complaints(user):

    if not user or user["role"] != "citizen":
        return pd.DataFrame()

    df = pd.read_sql_query(
        "SELECT complaint_id, category, description, location, priority, status, date "
        "FROM complaints WHERE user_id=? ORDER BY complaint_id DESC",
        conn,
        params=(user["user_id"],)
    )

    return df


# Authority dashboard
def view_all_complaints(user):

    if not user or user["role"] != "authority":
        return pd.DataFrame(), "Please login as an authority."

    df = pd.read_sql_query("""
    SELECT complaint_id, user_id, category, description,
           location, priority, status, date
    FROM complaints
    ORDER BY complaint_id DESC
    """, conn)

    return df, f"Total complaints: {len(df)}"


# Update complaint status
def update_status(user, complaint_id, new_status):

    if not user or user["role"] != "authority":
        return "Authority login required."

    try:
        cid = int(complaint_id)

        cursor.execute("""
        UPDATE complaints
        SET status=?
        WHERE complaint_id=?
        """, (new_status, cid))

        conn.commit()

        if cursor.rowcount == 0:
            return "Complaint ID not found."

        return f"Complaint SF-{cid:04d} updated to {new_status}."

    except ValueError:
        return "Enter a valid complaint ID."


# Build application
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    session = gr.State(None)

    gr.Markdown("""
    # 🏙️ SmartFix
    ### AI-Powered Public Issue Reporting and Resolution System

    Report public problems. Track complaints. Build cleaner communities.
    """)

    with gr.Tab("Register"):

        gr.Markdown("### Citizen Registration")

        reg_name = gr.Textbox(label="Full Name")
        reg_id = gr.Textbox(label="Create User ID")
        reg_pass = gr.Textbox(label="Create Password", type="password")

        reg_btn = gr.Button("Register", variant="primary")
        reg_msg = gr.Textbox(label="Registration Status")

        reg_btn.click(
            user_register,
            [reg_name, reg_id, reg_pass],
            reg_msg
        )

    with gr.Tab("Login"):

        login_id = gr.Textbox(label="User ID")
        login_pass = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login", variant="primary")
        login_msg = gr.Textbox(label="Login Status")

        login_btn.click(
            user_login,
            [login_id, login_pass],
            [session, login_msg]
        )

        gr.Markdown("""
        **Authority Demo Login**

        User ID: `admin`

        Password: `admin123`
        """)

    with gr.Tab("Citizen Dashboard"):

        gr.Markdown("### Submit a Public Complaint")

        category = gr.Dropdown(
            choices=[
                "Water Leakage",
                "Garbage Overflow",
                "Electricity Problem",
                "Drainage Problem",
                "Public Sanitation",
                "Road Damage",
                "Other"
            ],
            label="Select Problem Category"
        )

        description = gr.Textbox(
            label="Describe the Problem",
            lines=4
        )

        location = gr.Textbox(
            label="Problem Location",
            placeholder="Enter area, street or landmark"
        )

        image = gr.Image(
            label="Upload Problem Photograph",
            type="filepath"
        )

        submit_btn = gr.Button(
            "Submit Complaint",
            variant="primary"
        )

        submit_msg = gr.Textbox(
            label="Complaint Result",
            lines=5
        )

        submit_btn.click(
            submit_complaint,
            [session, category, description, location, image],
            submit_msg
        )

        gr.Markdown("### My Complaints")

        my_btn = gr.Button("View My Complaints")
        my_table = gr.Dataframe()

        my_btn.click(
            view_my_complaints,
            session,
            my_table
        )

    with gr.Tab("Authority Dashboard"):

        gr.Markdown("### Public Complaint Management")

        refresh_btn = gr.Button("View All Complaints")

        authority_table = gr.Dataframe()
        authority_count = gr.Textbox(label="Complaint Summary")

        refresh_btn.click(
            view_all_complaints,
            session,
            [authority_table, authority_count]
        )

        gr.Markdown("### Update Complaint Status")

        complaint_id = gr.Number(
            label="Complaint ID (Enter number only)",
            precision=0
        )

        new_status = gr.Dropdown(
            choices=["Pending", "In Progress", "Resolved"],
            label="New Status"
        )

        update_btn = gr.Button("Update Status")

        update_msg = gr.Textbox(label="Update Result")

        update_btn.click(
            update_status,
            [session, complaint_id, new_status],
            update_msg
        )

demo.launch(share=True)

/tmp/ipykernel_1130/143783435.py:184: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4805a33b2027f9cdf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Training dataset
training_complaints = [
    "Water is leaking from broken pipes",
    "Continuous water wastage from pipeline",
    "No water supply in our area",
    "Tap water is overflowing",

    "Garbage bins are overflowing",
    "Waste is scattered on the road",
    "Dustbins are not cleaned",
    "Garbage is collected irregularly",

    "Streetlights are not working",
    "Electric poles have damaged wires",
    "Frequent electricity problems",
    "Street lights are broken",

    "Drain water is overflowing",
    "Sewage water is spreading on roads",
    "Drainage pipes are blocked",
    "Dirty water is stagnating",

    "Public toilets are very dirty",
    "Bad smell in public toilets",
    "Public places are unhygienic",
    "Toilets are not maintained",

    "Roads have large potholes",
    "Road surface is damaged",
    "Cracks on the main road",
    "Roads need repair"
]

training_categories = (
    ["Water Leakage"] * 4 +
    ["Garbage Overflow"] * 4 +
    ["Electricity Problem"] * 4 +
    ["Drainage Problem"] * 4 +
    ["Public Sanitation"] * 4 +
    ["Road Damage"] * 4
)

# Create AI model
ai_classifier = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", MultinomialNB())
])

# Train model
ai_classifier.fit(training_complaints, training_categories)

print("AI Complaint Classification Model Trained Successfully!")

AI Complaint Classification Model Trained Successfully!


In [7]:
test_complaints = [
    "Water is continuously leaking near my house",
    "Garbage is overflowing near the bus stop",
    "Streetlights are not working at night",
    "The drainage system is blocked"
]

predictions = ai_classifier.predict(test_complaints)

for complaint, prediction in zip(test_complaints, predictions):
    print("Complaint:", complaint)
    print("AI Category:", prediction)
    print("-----------------------------")

Complaint: Water is continuously leaking near my house
AI Category: Water Leakage
-----------------------------
Complaint: Garbage is overflowing near the bus stop
AI Category: Garbage Overflow
-----------------------------
Complaint: Streetlights are not working at night
AI Category: Electricity Problem
-----------------------------
Complaint: The drainage system is blocked
AI Category: Drainage Problem
-----------------------------


In [8]:
demo.close()
print("Previous application stopped successfully!")

Closing server running on port: 7861
Previous application stopped successfully!


In [9]:
def submit_complaint(user, description, location, image):

    if not user or user["role"] != "citizen":
        return "Please login as a citizen first."

    if not description or not location:
        return "Please enter complaint description and location."

    # AI automatically predicts category
    prediction = ai_classifier.predict([description])[0]

    # Save uploaded photograph
    image_path = None

    if image:
        extension = os.path.splitext(image)[1] or ".png"
        filename = (
            f"{user['user_id']}_"
            f"{datetime.now().strftime('%Y%m%d%H%M%S%f')}{extension}"
        )

        image_path = os.path.join("uploads", filename)
        shutil.copy(image, image_path)

    # Store complaint
    cursor.execute("""
    INSERT INTO complaints
    (user_id, category, description, location, priority, status, image, date)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        user["user_id"],
        prediction,
        description,
        location,
        "Pending AI Analysis",
        "Pending",
        image_path,
        datetime.now().strftime("%Y-%m-%d %H:%M")
    ))

    conn.commit()

    complaint_id = cursor.lastrowid

    return f"""
    COMPLAINT SUBMITTED SUCCESSFULLY!

    Complaint ID: SF-{complaint_id:04d}

    AI Predicted Category: {prediction}

    Location: {location}

    Status: Pending
    """


print("AI complaint submission function integrated successfully!")

AI complaint submission function integrated successfully!


In [10]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    session = gr.State(None)

    gr.Markdown("""
    # 🏙️ SmartFix
    ### AI-Powered Public Issue Reporting and Resolution System
    Report public problems. Track progress. Improve your community.
    """)

    # REGISTRATION
    with gr.Tab("Register"):
        gr.Markdown("### Citizen Registration")

        reg_name = gr.Textbox(label="Full Name")
        reg_id = gr.Textbox(label="Choose User ID")
        reg_password = gr.Textbox(label="Password", type="password")

        reg_btn = gr.Button("Register")
        reg_msg = gr.Textbox(label="Registration Status")

        reg_btn.click(
            user_register,
            [reg_name, reg_id, reg_password],
            reg_msg
        )

    # LOGIN
    with gr.Tab("Login"):
        gr.Markdown("### Login to SmartFix")

        login_id = gr.Textbox(label="User ID")
        login_password = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login")
        login_msg = gr.Textbox(label="Login Status")

        login_btn.click(
            user_login,
            [login_id, login_password],
            [session, login_msg]
        )

        gr.Markdown("""
        **Authority Demo Login**

        User ID: `admin`

        Password: `admin123`
        """)

    # CITIZEN DASHBOARD
    with gr.Tab("Citizen Dashboard"):

        gr.Markdown("### Submit a Public Complaint")

        description = gr.Textbox(
            label="Describe Your Problem",
            placeholder="Example: Water is continuously leaking near my house",
            lines=3
        )

        location = gr.Textbox(
            label="Problem Location",
            placeholder="Enter area or location"
        )

        image = gr.Image(
            label="Upload Problem Photograph (Optional)",
            type="filepath"
        )

        submit_btn = gr.Button("Submit Complaint")
        submit_msg = gr.Textbox(label="Submission Result", lines=6)

        submit_btn.click(
            submit_complaint,
            [session, description, location, image],
            submit_msg
        )

        gr.Markdown("### Track Your Complaints")

        view_btn = gr.Button("View My Complaints")
        my_complaints = gr.Dataframe()

        view_btn.click(
            view_my_complaints,
            session,
            my_complaints
        )

    # AUTHORITY DASHBOARD
    with gr.Tab("Authority Dashboard"):

        gr.Markdown("### Manage Public Complaints")

        refresh_btn = gr.Button("View All Complaints")
        all_complaints = gr.Dataframe()
        summary = gr.Textbox(label="Complaint Summary")

        refresh_btn.click(
            view_all_complaints,
            session,
            [all_complaints, summary]
        )

        gr.Markdown("### Update Complaint Status")

        complaint_id = gr.Number(
            label="Complaint ID (Enter Number Only)",
            precision=0
        )

        new_status = gr.Dropdown(
            ["Pending", "In Progress", "Resolved"],
            label="New Status"
        )

        update_btn = gr.Button("Update Status")
        update_msg = gr.Textbox(label="Update Result")

        update_btn.click(
            update_status,
            [session, complaint_id, new_status],
            update_msg
        )

demo.launch(share=True)

/tmp/ipykernel_1130/545194981.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://72bb36ce6daf09cd04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
additional_priority_data = [

    # HIGH PRIORITY
    ("Electric transformer caught fire", "High"),
    ("Live electrical wire fallen on the road", "High"),
    ("Flood water entering houses due to drainage failure", "High"),
    ("Drinking water contaminated across the locality", "High"),
    ("Main water pipeline burst causing severe flooding", "High"),
    ("Open manhole causing danger to pedestrians", "High"),
    ("Large tree fallen and blocking an emergency route", "High"),
    ("Sewage water entering drinking water pipeline", "High"),
    ("Electric pole collapsed near a crowded market", "High"),
    ("Severe flooding near a hospital entrance", "High"),
    ("Road collapsed creating a dangerous deep hole", "High"),
    ("Electrical cables sparking near a school", "High"),
    ("Drainage overflow spreading into residential houses", "High"),
    ("Broken water pipe flooding the entire street", "High"),
    ("Heavy garbage blocking access to a hospital", "High"),
    ("Dangerous exposed wires near a children's playground", "High"),
    ("Manhole cover missing on a busy highway", "High"),
    ("Sewage flooding a crowded public area", "High"),
    ("Streetlight pole fallen onto moving traffic", "High"),
    ("Major water contamination affecting many households", "High"),

    # MEDIUM PRIORITY
    ("Garbage accumulating near apartment buildings", "Medium"),
    ("Drain water overflowing near shops", "Medium"),
    ("Several potholes on the main road", "Medium"),
    ("Streetlights not functioning in residential streets", "Medium"),
    ("Water leaking from a damaged roadside pipe", "Medium"),
    ("Public toilet has not been cleaned for several days", "Medium"),
    ("Garbage collection vehicle not visiting regularly", "Medium"),
    ("Drainage water stagnating near houses", "Medium"),
    ("Broken pavement affecting pedestrians", "Medium"),
    ("Public dustbin overflowing with waste", "Medium"),
    ("Water tank overflowing in a public facility", "Medium"),
    ("Roadside garbage attracting stray animals", "Medium"),
    ("Blocked stormwater drain during rainy weather", "Medium"),
    ("Multiple streetlights stopped working in a park", "Medium"),
    ("Damaged manhole cover on a residential street", "Medium"),
    ("Public toilet has a broken water tap", "Medium"),
    ("Waste dumped near a school entrance", "Medium"),
    ("Road surface becoming uneven near a junction", "Medium"),
    ("Dirty water collecting near a bus stop", "Medium"),
    ("Garbage not removed from a public park", "Medium"),

    # LOW PRIORITY
    ("Small amount of litter near a roadside tree", "Low"),
    ("Minor crack on a park walking path", "Low"),
    ("One garden tap dripping occasionally", "Low"),
    ("Dustbin requires routine maintenance", "Low"),
    ("Faded road markings in a quiet street", "Low"),
    ("Small patch of grass needs trimming", "Low"),
    ("Minor paint peeling from a public wall", "Low"),
    ("One park bench requires repainting", "Low"),
    ("Small amount of leaves inside a roadside drain", "Low"),
    ("Minor damage to a public notice board", "Low"),
    ("One decorative light flickering in a garden", "Low"),
    ("Small litter near a public seating area", "Low"),
    ("Routine cleaning needed around a park bench", "Low"),
    ("Minor footpath tile discoloration", "Low"),
    ("Small amount of dust near a bus shelter", "Low"),
    ("One public dustbin needs a replacement lid", "Low"),
    ("Minor damage to a garden fence", "Low"),
    ("Small water drip from a garden sprinkler", "Low"),
    ("Routine maintenance needed for park plants", "Low"),
    ("Minor cleanliness issue near a public bench", "Low")
]

# Initialize the original dataset if it does not exist
if "priority_training_data" not in globals():
    priority_training_data = []

# Add the additional examples only once
if "additional_priority_data" not in globals() or not globals().get("additional_data_added", False):
    priority_training_data.extend(additional_priority_data)
    additional_data_added = True

print("Total training examples:", len(priority_training_data))
print("High:", sum(1 for _, p in priority_training_data if p == "High"))
print("Medium:", sum(1 for _, p in priority_training_data if p == "Medium"))
print("Low:", sum(1 for _, p in priority_training_data if p == "Low"))

Total training examples: 60
High: 20
Medium: 20
Low: 20


In [13]:
original_priority_data = [

    # HIGH
    ("Major water pipeline burst flooding nearby houses", "High"),
    ("Electric wires hanging dangerously over the road", "High"),
    ("Sewage water flooding residential buildings", "High"),
    ("Large pothole causing multiple road accidents", "High"),
    ("Fire caused by damaged electrical equipment", "High"),
    ("Contaminated water supply affecting an entire area", "High"),
    ("Drainage overflow entering people's homes", "High"),
    ("Broken electric pole leaning towards houses", "High"),
    ("Major water leakage flooding the main road", "High"),
    ("Open manhole creating a serious accident risk", "High"),
    ("Electrical sparks coming from a street pole", "High"),
    ("Heavy sewage overflow near a school", "High"),
    ("Road completely blocked due to a fallen tree", "High"),
    ("Water supply pipe severely damaged", "High"),
    ("Dangerous exposed electrical cables on footpath", "High"),
    ("Flooding caused by completely blocked drainage", "High"),
    ("Large damaged road causing traffic accidents", "High"),
    ("Public toilet sewage overflowing into nearby shops", "High"),
    ("Streetlight pole fallen across the road", "High"),
    ("Major garbage accumulation blocking emergency access", "High"),

    # MEDIUM
    ("Garbage bins overflowing near the bus stop", "Medium"),
    ("Drainage water collecting on the street", "Medium"),
    ("Several streetlights not working in the area", "Medium"),
    ("Road surface damaged with multiple potholes", "Medium"),
    ("Continuous water leakage from a public tap", "Medium"),
    ("Garbage not collected for several days", "Medium"),
    ("Blocked drainage causing bad smell", "Medium"),
    ("Public toilet requires timely cleaning", "Medium"),
    ("Broken streetlight near a residential street", "Medium"),
    ("Water overflowing from a community tank", "Medium"),
    ("Waste dumped near a public park", "Medium"),
    ("Cracks appearing on a busy road", "Medium"),
    ("Drain cover damaged on a footpath", "Medium"),
    ("Irregular garbage collection in residential area", "Medium"),
    ("Dirty water stagnating near public facilities", "Medium"),
    ("Multiple potholes affecting vehicle movement", "Medium"),
    ("Public dustbins are not being maintained", "Medium"),
    ("Water pipeline leaking continuously on roadside", "Medium"),
    ("Public toilet water supply not working", "Medium"),
    ("Drainage system partially blocked", "Medium"),

    # LOW
    ("Small water leak from a garden tap", "Low"),
    ("Minor crack on the roadside", "Low"),
    ("One decorative streetlight not working", "Low"),
    ("Small amount of litter near a park", "Low"),
    ("Dustbin needs routine cleaning", "Low"),
    ("Minor dripping from a public tap", "Low"),
    ("Small patch of damaged pavement", "Low"),
    ("Occasional bad smell near a dustbin", "Low"),
    ("One streetlight flickering occasionally", "Low"),
    ("Minor paint damage on a public bench", "Low"),
    ("Small amount of leaves blocking a drain", "Low"),
    ("Public park requires routine maintenance", "Low"),
    ("Minor road marking damage", "Low"),
    ("One small garbage bag left near a bin", "Low"),
    ("Small crack in a footpath tile", "Low"),
    ("Routine cleaning required at a bus stop", "Low"),
    ("Minor water dripping from a pipe", "Low"),
    ("Small area of grass needs maintenance", "Low"),
    ("One public dustbin lid is damaged", "Low"),
    ("Minor cleanliness issue near a public facility", "Low")
]

# Combine both datasets
priority_training_data = original_priority_data + additional_priority_data

print("Total training examples:", len(priority_training_data))
print("High:", sum(1 for _, p in priority_training_data if p == "High"))
print("Medium:", sum(1 for _, p in priority_training_data if p == "Medium"))
print("Low:", sum(1 for _, p in priority_training_data if p == "Low"))

Total training examples: 120
High: 40
Medium: 40
Low: 40


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Separate complaints and priority labels
X = [complaint for complaint, priority in priority_training_data]
y = [priority for complaint, priority in priority_training_data]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create and train AI model
priority_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("model", MultinomialNB())
])

priority_model.fit(X_train, y_train)

# Evaluate model
predictions = priority_model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("AI Priority Model Trained Successfully!")
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

AI Priority Model Trained Successfully!
Training examples: 96
Testing examples: 24
Model Accuracy: 79.17 %

Classification Report:
              precision    recall  f1-score   support

        High       0.78      0.88      0.82         8
         Low       0.88      0.88      0.88         8
      Medium       0.71      0.62      0.67         8

    accuracy                           0.79        24
   macro avg       0.79      0.79      0.79        24
weighted avg       0.79      0.79      0.79        24



In [15]:
for complaint, actual, predicted in zip(
    X_test, y_test, predictions
):
    if actual != predicted:
        print("Complaint:", complaint)
        print("Actual Priority:", actual)
        print("Predicted Priority:", predicted)
        print("-" * 50)

Complaint: Water pipeline leaking continuously on roadside
Actual Priority: Medium
Predicted Priority: High
--------------------------------------------------
Complaint: Electrical sparks coming from a street pole
Actual Priority: High
Predicted Priority: Medium
--------------------------------------------------
Complaint: Several potholes on the main road
Actual Priority: Medium
Predicted Priority: High
--------------------------------------------------
Complaint: Public toilet requires timely cleaning
Actual Priority: Medium
Predicted Priority: Low
--------------------------------------------------
Complaint: Faded road markings in a quiet street
Actual Priority: Low
Predicted Priority: Medium
--------------------------------------------------


In [16]:
improvement_data = [

    # HIGH PRIORITY
    ("Live electrical wires producing sparks", "High"),
    ("Electric pole sparking and creating fire risk", "High"),
    ("Dangerous electrical short circuit on public road", "High"),
    ("Exposed high voltage wires near pedestrians", "High"),
    ("Electrical transformer emitting sparks and smoke", "High"),

    # MEDIUM PRIORITY
    ("Water pipeline leaking continuously along roadside", "Medium"),
    ("Several potholes affecting vehicles on the main road", "Medium"),
    ("Public toilet needs regular cleaning and maintenance", "Medium"),
    ("Water leaking from a public supply pipe without flooding", "Medium"),
    ("Multiple streetlights not functioning in a residential area", "Medium"),

    # LOW PRIORITY
    ("Faded road markings on a quiet residential street", "Low"),
    ("Minor paint fading on roadside markings", "Low"),
    ("Small decorative light flickering in a public garden", "Low"),
    ("Minor crack on an unused footpath", "Low"),
    ("Routine painting required on a public bench", "Low")
]

# Add examples only once
if not globals().get("improvement_data_added", False):
    priority_training_data.extend(improvement_data)
    improvement_data_added = True

print("Updated training examples:", len(priority_training_data))

Updated training examples: 135


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Prepare all training data
X = [complaint for complaint, priority in priority_training_data]
y = [priority for complaint, priority in priority_training_data]

# Train using all 135 examples
priority_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("model", MultinomialNB())
])

priority_model.fit(X, y)

print("Improved AI Priority Model Trained!")
print("Total training examples:", len(X))

# New unseen test complaints
test_complaints = [
    "High voltage cable has fallen near a crowded market",
    "A major sewage leak is flooding residential houses",
    "Electric transformer is producing smoke",

    "Garbage has not been collected for several days",
    "Water is leaking continuously from a roadside pipeline",
    "Several potholes are affecting traffic movement",
    "Public toilet requires regular maintenance",
    "Multiple streetlights are not working",

    "A small garden tap is dripping",
    "Minor paint damage on a public bench",
    "Faded markings on a quiet road",
    "Small litter near a park bench"
]

actual_priorities = [
    "High", "High", "High",
    "Medium", "Medium", "Medium", "Medium", "Medium",
    "Low", "Low", "Low", "Low"
]

# Predict priorities
predicted_priorities = priority_model.predict(test_complaints)

print("\nNEW COMPLAINT TEST RESULTS")

for complaint, actual, predicted in zip(
    test_complaints, actual_priorities, predicted_priorities
):
    print("\nComplaint:", complaint)
    print("Expected:", actual)
    print("AI Prediction:", predicted)

print("\nTest Accuracy:",
      round(accuracy_score(actual_priorities, predicted_priorities) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(actual_priorities, predicted_priorities))

Improved AI Priority Model Trained!
Total training examples: 135

NEW COMPLAINT TEST RESULTS

Complaint: High voltage cable has fallen near a crowded market
Expected: High
AI Prediction: High

Complaint: A major sewage leak is flooding residential houses
Expected: High
AI Prediction: High

Complaint: Electric transformer is producing smoke
Expected: High
AI Prediction: High

Complaint: Garbage has not been collected for several days
Expected: Medium
AI Prediction: Medium

Complaint: Water is leaking continuously from a roadside pipeline
Expected: Medium
AI Prediction: Medium

Complaint: Several potholes are affecting traffic movement
Expected: Medium
AI Prediction: Medium

Complaint: Public toilet requires regular maintenance
Expected: Medium
AI Prediction: Medium

Complaint: Multiple streetlights are not working
Expected: Medium
AI Prediction: Medium

Complaint: A small garden tap is dripping
Expected: Low
AI Prediction: Low

Complaint: Minor paint damage on a public bench
Expected: L

In [18]:
demo.close()
print("Previous SmartFix application stopped!")

Closing server running on port: 7861
Previous SmartFix application stopped!


In [19]:
def submit_complaint(user, description, location, image):

    if not user or user["role"] != "citizen":
        return "Please login as a citizen first."

    if not description or not location:
        return "Please enter complaint description and location."

    # AI MODEL 1: Predict complaint category
    category_prediction = ai_classifier.predict([description])[0]

    # AI MODEL 2: Predict complaint priority
    priority_prediction = priority_model.predict([description])[0]

    # Save uploaded photograph
    image_path = None

    if image:
        extension = os.path.splitext(image)[1] or ".png"
        filename = (
            f"{user['user_id']}_"
            f"{datetime.now().strftime('%Y%m%d%H%M%S%f')}{extension}"
        )

        image_path = os.path.join("uploads", filename)
        shutil.copy(image, image_path)

    # Save complaint in database
    cursor.execute("""
    INSERT INTO complaints
    (user_id, category, description, location, priority, status, image, date)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        user["user_id"],
        category_prediction,
        description,
        location,
        priority_prediction,
        "Pending",
        image_path,
        datetime.now().strftime("%Y-%m-%d %H:%M")
    ))

    conn.commit()

    complaint_id = cursor.lastrowid

    return f"""
    COMPLAINT SUBMITTED SUCCESSFULLY!

    Complaint ID: SF-{complaint_id:04d}

    AI Predicted Category: {category_prediction}

    AI Predicted Priority: {priority_prediction}

    Location: {location}

    Status: Pending
    """

print("Both AI models integrated successfully!")

Both AI models integrated successfully!


In [20]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    session = gr.State(None)

    gr.Markdown("""
    # 🏙️ SmartFix
    ### AI-Powered Public Issue Reporting and Resolution System
    """)

    with gr.Tab("Register"):
        gr.Markdown("### Citizen Registration")

        reg_name = gr.Textbox(label="Full Name")
        reg_id = gr.Textbox(label="Choose User ID")
        reg_password = gr.Textbox(label="Password", type="password")

        reg_btn = gr.Button("Register")
        reg_msg = gr.Textbox(label="Registration Status")

        reg_btn.click(
            user_register,
            [reg_name, reg_id, reg_password],
            reg_msg
        )

    with gr.Tab("Login"):
        gr.Markdown("### Login to SmartFix")

        login_id = gr.Textbox(label="User ID")
        login_password = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login")
        login_msg = gr.Textbox(label="Login Status")

        login_btn.click(
            user_login,
            [login_id, login_password],
            [session, login_msg]
        )

        gr.Markdown("Authority Login: admin / admin123")

    with gr.Tab("Citizen Dashboard"):
        gr.Markdown("### Submit a Public Complaint")

        description = gr.Textbox(
            label="Describe Your Problem",
            lines=3
        )

        location = gr.Textbox(label="Problem Location")

        image = gr.Image(
            label="Upload Photograph (Optional)",
            type="filepath"
        )

        submit_btn = gr.Button("Submit Complaint")
        submit_msg = gr.Textbox(
            label="AI Analysis Result",
            lines=8
        )

        submit_btn.click(
            submit_complaint,
            [session, description, location, image],
            submit_msg
        )

        gr.Markdown("### Track Your Complaints")

        view_btn = gr.Button("View My Complaints")
        my_complaints = gr.Dataframe()

        view_btn.click(
            view_my_complaints,
            session,
            my_complaints
        )

    with gr.Tab("Authority Dashboard"):
        gr.Markdown("### Manage Public Complaints")

        refresh_btn = gr.Button("View All Complaints")
        all_complaints = gr.Dataframe()
        summary = gr.Textbox(label="Complaint Summary")

        refresh_btn.click(
            view_all_complaints,
            session,
            [all_complaints, summary]
        )

        gr.Markdown("### Update Complaint Status")

        complaint_id = gr.Number(
            label="Complaint ID",
            precision=0
        )

        new_status = gr.Dropdown(
            ["Pending", "In Progress", "Resolved"],
            label="New Status"
        )

        update_btn = gr.Button("Update Status")
        update_msg = gr.Textbox(label="Update Result")

        update_btn.click(
            update_status,
            [session, complaint_id, new_status],
            update_msg
        )

demo.launch(share=True)

/tmp/ipykernel_1130/2656375920.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58ad1458ae0d71f4a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
# Fetch all existing complaints
cursor.execute("""
SELECT complaint_id, description
FROM complaints
""")

old_complaints = cursor.fetchall()

# Reanalyse each complaint using both AI models
for complaint_id, description in old_complaints:

    predicted_category = ai_classifier.predict([description])[0]
    predicted_priority = priority_model.predict([description])[0]

    cursor.execute("""
    UPDATE complaints
    SET category = ?, priority = ?
    WHERE complaint_id = ?
    """, (
        predicted_category,
        predicted_priority,
        complaint_id
    ))

conn.commit()

print("Old complaints successfully updated!")
print("Total complaints reanalysed:", len(old_complaints))

Old complaints successfully updated!
Total complaints reanalysed: 5


In [22]:
pd.read_sql_query("""
SELECT complaint_id, description, category, priority, status
FROM complaints
""", conn)

,complaint_id,description,category,priority,status
0,1,Water is continuously leaking from a damaged p...,Water Leakage,Medium,In Progress
1,2,Water is continuously leaking near my house,Water Leakage,Medium,Resolved
2,3,Garbage is overflowing near the bus stop,Garbage Overflow,Medium,Pending
3,4,Streetlights are not working at night,Electricity Problem,Medium,Pending
4,5,The drainage system is blocked,Drainage Problem,Medium,Pending


In [23]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58ad1458ae0d71f4a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
